In [1]:
# bfe
import pandas as pd
import os
import os.path as osp
import numpy as np
import json
from datetime import datetime
from zoneinfo import ZoneInfo
import requests

In [2]:

try:
    PROJECT_ROOT = osp.abspath(osp.join(osp.dirname(__file__),'..'))
except:
    PROJECT_ROOT = osp.abspath(osp.join(os.getcwd(),'..'))

print(PROJECT_ROOT)
STATIONS_FILE = osp.join(PROJECT_ROOT,'src','stations_infos.json')

/home/bfeldmann/projects/formation_mlops/projet_meteoAUS


In [ ]:
class DailyWeatherDATA:
    def __init__(self):
        self.url_daily_weather = 'https://www.bom.gov.au/climate/dwo/{year_month}/text/{bom_id}.{year_month}.csv'

        with open(STATIONS_FILE) as src:
            self.stations_infos = json.load(src)
            
    def get_url(self,city, time='last'):
        station_info = self.stations_infos.get(city,None)
        
        if station_info is None:
            raise ValueError(f"Unknown city: {city}")
        
        if time=='last':
            now = datetime.now(ZoneInfo(station_info['timezone']))
            date_month = now.strftime("%Y%m")
        elif len(time)==6 and time[0:2]=='20':
            date_month = time
        else:
            raise ValueError(f'Not valid time, expected "yearmonth" but got {time}')
        
        return self.url_daily_weather.format(year_month=date_month, bom_id=station_info['bom_id'])
    
    def get_report(self,csv_url, out_path):
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(csv_url, headers=headers)
        response.raise_for_status()

        with open(out_path, "wb") as f:
            f.write(response.content)
        return True

    def cleaning_report(self,in_file, out_file):
        raw_cols = ['Date', 'Minimum temperature (°C)', 'Maximum temperature (°C)',
                    'Rainfall (mm)', 'Evaporation (mm)', 'Sunshine (hours)',
                    'Direction of maximum wind gust ', 'Speed of maximum wind gust (km/h)',
                    'Time of maximum wind gust', '9am Temperature (°C)',
                    '9am relative humidity (%)', '9am cloud amount (oktas)',
                    '9am wind direction', '9am wind speed (km/h)', '9am MSL pressure (hPa)',
                    '3pm Temperature (°C)', '3pm relative humidity (%)',
                    '3pm cloud amount (oktas)', '3pm wind direction',
                    '3pm wind speed (km/h)', '3pm MSL pressure (hPa)']

        dict_cols = {
            'Date':'datetime', 'MinTemp': 'float32', 'MaxTemp': 'float32', 'Rainfall': 'float32', 'Evaporation': 'float32',
            'Sunshine': 'float32', 'WindGustDir': 'string', 'WindGustSpeed': 'float32', 'TimeMaxWindGust': 'string',
            'Temp9am': 'float32', 'Humidity9am': 'int16', 'Cloud9am': 'float32', 'WindDir9am': 'string',
            'WindSpeed9am': 'float32', 'Pressure9am': 'float32', 'Temp3pm': 'float32', 'Humidity3pm': 'int16',
            'Cloud3pm': 'float32', 'WindDir3pm': 'string', 'WindSpeed3pm': 'float32', 'Pressure3pm': 'float32'
            }
        
        df = pd.read_csv(in_file, delimiter=',', skiprows=7, encoding='latin-1')
        df = df.drop(columns=df.columns[0])
        
        if all(df.columns == raw_cols):
            df.columns = dict_cols.keys()
            
        for col,dtype in dict_cols.items():
            if dtype in ('int16', 'float32'):
                df[col] = pd.to_numeric(df[col], errors='coerce').astype(dtype)
                
            elif dtype=='datetime':
                df[col] = pd.to_datetime(df[col], errors='coerce')
                
            else:
                df[col] = df[col].astype(dtype)
        
        df = df.replace(
            [pd.NA, None, "NaN", "nan", "NAN", "NA", "N/A", "", " ", "null", "None"],
            np.nan
        )
        df = df.drop(columns='TimeMaxWindGust')        
        df.to_csv(out_file)

# TEST

In [7]:
data_file = osp.join(PROJECT_ROOT,'data','weatherAUS.csv')
df = pd.read_csv(data_file)
np.unique(df['Location'])

array(['Adelaide', 'Albany', 'Albury', 'AliceSprings', 'BadgerysCreek',
       'Ballarat', 'Bendigo', 'Brisbane', 'Cairns', 'Canberra', 'Cobar',
       'CoffsHarbour', 'Dartmoor', 'Darwin', 'GoldCoast', 'Hobart',
       'Katherine', 'Launceston', 'Melbourne', 'MelbourneAirport',
       'Mildura', 'Moree', 'MountGambier', 'MountGinini', 'Newcastle',
       'Nhil', 'NorahHead', 'NorfolkIsland', 'Nuriootpa', 'PearceRAAF',
       'Penrith', 'Perth', 'PerthAirport', 'Portland', 'Richmond', 'Sale',
       'SalmonGums', 'Sydney', 'SydneyAirport', 'Townsville',
       'Tuggeranong', 'Uluru', 'WaggaWagga', 'Walpole', 'Watsonia',
       'Williamtown', 'Witchcliffe', 'Wollongong', 'Woomera'],
      dtype=object)